# `swarmtorch` GPU vs NumPy headline benchmark

Run this on Colab or Kaggle with a **GPU runtime** to validate the paper's headline claim:
PyTorch + GPU + vmap make metaheuristics dramatically faster than the equivalent
pure-NumPy reference, especially as `swarm_size` and `dim` grow.

**Colab:** Runtime → Change runtime type → Hardware accelerator: GPU (T4 or better).

**Kaggle:** Settings → Accelerator: GPU.

The notebook compares four implementations of identical PSO on the same synthetic
test functions under the same FE budget:

* `numpy` — pure-NumPy reference (the pyMetaheuristic-style baseline).
* `swarmtorch-cpu-loop` — swarmtorch.PSO on CPU with the legacy per-particle Python loop.
* `swarmtorch-cpu-vmap` — swarmtorch.PSO on CPU with `torch.func.functional_call` + `vmap`.
* `swarmtorch-cuda-vmap` — same vmap path on CUDA. **The headline.**

Expected result on a T4 / V100: 10–100× speedup at `swarm_size >= 256`.

## 1. Install

In [ ]:
# Clone the repo and install with benchmark + cmaes extras.
# Skip the clone step if you've already uploaded the repo to your Colab/Kaggle session.
import os
import sys
import subprocess

if not os.path.isdir('swarmtorch'):
    subprocess.check_call(['git', 'clone', 'https://github.com/hallelx2/swarmtorch.git'])
%cd swarmtorch/swarmtorch

# Install the package and benchmark dependencies. 
# (PyTorch is pre-installed on Colab / Kaggle.)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[benchmark,cmaes]'])

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Quick smoke run

Tiny grid (~30 seconds) — useful first to confirm everything works. Skip to the next cell for the real headline grid.

In [ ]:
!python scripts/run_gpu_vs_numpy.py --quick --output-dir results/gpu_quick

## 3. Headline benchmark grid

Sweep across 3 functions × 2 dimensions × 3 swarm sizes × 2 seeds × 4 variants.

Total ≈ 144 cells, ~10–30 minutes on a T4 GPU. Adjust `--max-fe` if you want it faster.

In [ ]:
!python scripts/run_gpu_vs_numpy.py \
    --output-dir results/gpu_vs_numpy \
    --functions sphere rastrigin ackley \
    --dims 100 1000 \
    --swarm-sizes 64 256 1024 \
    --seeds 0 1 \
    --max-fe 20000

## 4. View the speedup report

In [ ]:
from IPython.display import Markdown
Markdown(open('results/gpu_vs_numpy/report.md').read())

## 5. Plot the speedup curve

One subplot per function. X-axis = swarm size. Y-axis = wall-clock seconds (log scale).
Each line is a variant; the gap between `numpy` and `swarmtorch-cuda-vmap` is the headline.


In [ ]:
import json
import glob
from collections import defaultdict
import matplotlib.pyplot as plt

records = [json.load(open(p)) for p in sorted(glob.glob('results/gpu_vs_numpy/*.json'))]

# Group by (function, dim, variant) -> list of (swarm_size, wall_seconds).
g = defaultdict(list)
for r in records:
    g[(r['function'], r['dim'], r['variant'])].append((r['swarm_size'], r['wall_seconds']))

funcs = sorted({(f, d) for (f, d, _) in g})
fig, axes = plt.subplots(1, len(funcs), figsize=(5 * len(funcs), 4), squeeze=False, dpi=120)
for ax, (func, dim) in zip(axes[0], funcs):
    for variant in ['numpy', 'swarmtorch-cpu-loop', 'swarmtorch-cpu-vmap', 'swarmtorch-cuda-vmap']:
        if (func, dim, variant) not in g:
            continue
        pts = sorted(g[(func, dim, variant)])
        # Average across seeds.
        from collections import defaultdict as dd
        avg = dd(list)
        for ss, w in pts:
            avg[ss].append(w)
        xs = sorted(avg)
        ys = [sum(avg[x]) / len(avg[x]) for x in xs]
        ax.plot(xs, ys, marker='o', label=variant)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('swarm size')
    ax.set_ylabel('wall-clock seconds')
    ax.set_title(f'{func}, d={dim}')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, loc='best')
fig.tight_layout()
fig.savefig('results/gpu_vs_numpy/speedup_curves.png', bbox_inches='tight')
plt.show()
print('Saved: results/gpu_vs_numpy/speedup_curves.png')

## 6. (Optional) Download results

Bundle JSONs + report + plot for use in the paper / drag back to your laptop.

In [ ]:
import shutil
shutil.make_archive('gpu_vs_numpy_results', 'zip', 'results/gpu_vs_numpy')
print('Bundled: gpu_vs_numpy_results.zip')
# In Colab:
try:
    from google.colab import files
    files.download('gpu_vs_numpy_results.zip')
except ImportError:
    pass